# AuthSignal Project Part 1 — Dataset Exploration, Cleaning, Visualization, and First Model

**Project:** AuthSignal: AI-Based Denial Risk and Denial Reason Prediction for Prior Authorization Workflows  
**Part 1 focus:** Denial Risk Prediction dataset  

This notebook covers the required Part 1 tasks: dataset description, feature inspection, preprocessing with pandas, visualization with matplotlib/seaborn, methodology, and one working machine learning model.

## 1. Dataset Description

The dataset used in this notebook is `authsignal_denial_risk_dataset_200000.csv`. It is a synthetic prior-authorization dataset generated for the AuthSignal project. The topic/domain is healthcare administration, specifically prior authorization approval/denial prediction.

The dataset contains synthetic case-level records with payer information, patient/request features, document availability indicators, policy-match scores, and a target label `denial_outcome` with two classes: `Approved` and `Denied`.

The data was generated using realistic feature logic inspired by public healthcare data sources such as Synthea synthetic patient records, CMS synthetic claims concepts, CMS Medicare Coverage Database policy logic, and CMS Part D formulary indicators such as prior authorization, step therapy, and quantity limits. No real patient data is used.

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

## 3. Load the Data

In [ ]:
DATA_PATH = '../data/authsignal_denial_risk_dataset_200000.csv'
df = pd.read_csv(DATA_PATH)
df.head()

## 4. Dimensionality

In [ ]:
print('Rows, Columns:', df.shape)

## 5. Feature Overview and Data Types

In [ ]:
feature_info = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str),
    'missing_values': df.isna().sum().values,
    'missing_percent': (df.isna().mean().values * 100).round(2),
    'unique_values': [df[c].nunique(dropna=True) for c in df.columns]
})
feature_info

### Main Features

Important features include:

- `payer_family`, `payer_name`, `plan_type`: categorical payer/plan features.
- `patient_age`, `age_group`: numerical and categorical patient age features.
- `request_type`, `diagnosis_category`, `provider_specialty`: clinical/request category features.
- `clinical_note_present`, `lab_report_attached`, `referral_attached`: document availability flags.
- `missing_required_conditions`: numerical count of missing criteria.
- `policy_match_score`, `note_completeness_score`: numerical scores from 0 to 1, with some dirty values intentionally inserted.
- `requested_amount_usd`, `requested_units`: request size/amount features.
- `denial_outcome`: target variable for the first model.

## 6. Missing Values

In [ ]:
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary[missing_summary > 0]

For this phase, missing numerical values will be imputed using median values and categorical missing values will be imputed using the most frequent value inside the model pipeline. For exploratory cleaning, we will also cap invalid score values and age outliers.

## 7. Descriptive Statistics

In [ ]:
df.describe(include='all').T.head(30)

## 8. Outlier Exploration

In [ ]:
numeric_cols_for_outliers = ['patient_age', 'supporting_doc_count', 'missing_required_conditions', 'policy_match_score', 'note_completeness_score', 'requested_units', 'requested_amount_usd']

for col in numeric_cols_for_outliers:
    plt.figure(figsize=(6, 3))
    plt.boxplot(df[col].dropna(), vert=False)
    plt.title(f'Boxplot of {col}')
    plt.xlabel(col)
    plt.show()

## 9. Basic Cleaning

Cleaning decisions:

- Invalid ages below 0 or above 100 are capped to the realistic range 0–100.
- `policy_match_score` and `note_completeness_score` are capped to 0–1 because they are score features.
- Extreme `requested_amount_usd` values are capped using the IQR method.
- Categorical missing values and numeric missing values are handled later inside the model pipeline.

In [ ]:
df_clean = df.copy()

# Cap age
df_clean['patient_age'] = pd.to_numeric(df_clean['patient_age'], errors='coerce').clip(lower=0, upper=100)

# Cap score columns to 0-1
for col in ['policy_match_score', 'note_completeness_score']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').clip(lower=0, upper=1)

# IQR capping for requested amount
amount_col = 'requested_amount_usd'
q1 = df_clean[amount_col].quantile(0.25)
q3 = df_clean[amount_col].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
df_clean[amount_col] = df_clean[amount_col].clip(lower=lower, upper=upper)

print('Cleaned shape:', df_clean.shape)
df_clean[numeric_cols_for_outliers].describe().T

## 10. Data Type Conversion

In [ ]:
binary_cols = [
    'prior_auth_required', 'pa_form_submitted', 'clinical_note_present', 'diagnosis_code_present',
    'procedure_code_present', 'lab_report_attached', 'referral_attached', 'imaging_report_attached',
    'previous_treatment_history_included', 'step_therapy_required', 'step_therapy_met',
    'quantity_limit_flag', 'quantity_limit_exceeded', 'non_covered_flag', 'duplicate_request_flag',
    'coding_mismatch_flag'
]

for col in binary_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

categorical_cols_preview = ['payer_family', 'payer_name', 'plan_type', 'age_group', 'provider_specialty', 'diagnosis_category', 'request_type', 'site_of_care', 'network_status', 'urgency']
for col in categorical_cols_preview:
    df_clean[col] = df_clean[col].astype('category')

df_clean.dtypes.head(25)

# 11. Data Visualization with Matplotlib

## 11.1 Numerical Feature Distributions

In [ ]:
for col in ['patient_age', 'supporting_doc_count', 'missing_required_conditions', 'policy_match_score', 'note_completeness_score', 'requested_amount_usd']:
    plt.figure(figsize=(6, 4))
    plt.hist(df_clean[col].dropna(), bins=30)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

## 11.2 Categorical Feature Distributions

In [ ]:
for col in ['denial_outcome', 'request_type', 'payer_family', 'project_denial_reason']:
    counts = df_clean[col].value_counts(dropna=False).head(12)
    plt.figure(figsize=(8, 4))
    plt.bar(counts.index.astype(str), counts.values)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 11.3 Scatter Plot: Policy Match vs Note Completeness

In [ ]:
sample_plot = df_clean.sample(5000, random_state=42)
colors = sample_plot['denial_outcome'].map({'Approved': 0, 'Denied': 1})
plt.figure(figsize=(7, 5))
plt.scatter(sample_plot['policy_match_score'], sample_plot['note_completeness_score'], c=colors, alpha=0.35)
plt.title('Policy Match Score vs Note Completeness Score')
plt.xlabel('Policy Match Score')
plt.ylabel('Note Completeness Score')
plt.show()

## 11.4 Correlation Matrix for Numerical Features

In [ ]:
num_for_corr = ['patient_age', 'supporting_doc_count', 'required_condition_count', 'missing_required_conditions', 'prior_denials_count', 'policy_match_score', 'note_completeness_score', 'payer_policy_strictness', 'requested_units', 'requested_amount_usd']
correlations = df_clean[num_for_corr].corr(numeric_only=True)
plt.figure(figsize=(9, 7))
plt.imshow(correlations, aspect='auto')
plt.colorbar()
plt.xticks(range(len(num_for_corr)), num_for_corr, rotation=90)
plt.yticks(range(len(num_for_corr)), num_for_corr)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

## 11.5 Pair Plot

The assignment mentions pair plots using seaborn. Pair plots are expensive on 200,000 rows, so we use a small sample.

In [ ]:
try:
    import seaborn as sns
    pair_cols = ['patient_age', 'missing_required_conditions', 'policy_match_score', 'note_completeness_score', 'requested_amount_usd', 'denial_outcome']
    sns.pairplot(df_clean[pair_cols].sample(1500, random_state=42), hue='denial_outcome', diag_kind='hist')
    plt.show()
except Exception as e:
    print('Seaborn pairplot skipped:', e)

# 12. Methodology

Based on the dataset exploration, the project will use at least two machine learning models:

1. **Denial Risk Predictor** — structured classification model using features such as missing required conditions, policy match score, note completeness score, payer type, request type, and document flags. Suitable models include Logistic Regression, Random Forest, and Gradient Boosting.
2. **Denial Reason Classifier** — NLP text classification model using denial letter text or short denial summary. Suitable models include TF-IDF + Logistic Regression and TF-IDF + Linear SVM.

An optional third model can be added:

3. **Document Completeness Predictor** — structured classification model that predicts whether the submitted document package is complete or incomplete. Suitable models include Logistic Regression, Decision Tree, and Random Forest.

For Part 1, this notebook shows the working of the first model: **Denial Risk Predictor** using Logistic Regression.

# 13. First Model: Denial Risk Predictor

## 13.1 Prepare Features and Target

To avoid leakage, columns that directly reveal the label or synthetic label-generation probability are removed before training.

In [ ]:
target = 'denial_outcome'
leakage_or_id_cols = [
    'case_id', 'denial_probability_true_synthetic', 'detailed_denial_reason', 'project_denial_reason',
    'diagnosis_code', 'procedure_code', 'ndc_code'
]

model_df = df_clean.drop(columns=leakage_or_id_cols, errors='ignore').copy()

X = model_df.drop(columns=[target])
y = (model_df[target] == 'Denied').astype(int)

# Use a sample for faster Part 1 execution on normal laptops. Increase this if your machine can handle it.
SAMPLE_SIZE = min(60000, len(X))
X_sample = X.sample(SAMPLE_SIZE, random_state=42)
y_sample = y.loc[X_sample.index]

X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42, stratify=y_sample
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Target distribution in sample:')
print(y_sample.value_counts(normalize=True).rename({0:'Approved', 1:'Denied'}))

## 13.2 Build Preprocessing + Logistic Regression Pipeline

In [ ]:
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=True)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', encoder)
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

clf = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

clf.fit(X_train, y_train)
print('Model trained successfully.')

## 13.3 Evaluate the First Model

In [ ]:
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('Precision:', round(precision_score(y_test, y_pred), 4))
print('Recall:', round(recall_score(y_test, y_pred), 4))
print('F1-score:', round(f1_score(y_test, y_pred), 4))
print('ROC-AUC:', round(roc_auc_score(y_test, y_proba), 4))
print('
Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('
Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Approved', 'Denied']))

## 13.4 Basic Explainability: Logistic Regression Coefficients

The following cell extracts influential features from the trained Logistic Regression model. Positive coefficients push the model toward `Denied`; negative coefficients push it toward `Approved`.

In [ ]:
# Get feature names after preprocessing
num_names = numeric_features
cat_encoder = clf.named_steps['preprocess'].named_transformers_['cat'].named_steps['onehot']
cat_names = cat_encoder.get_feature_names_out(categorical_features).tolist()
feature_names = num_names + cat_names

coefs = clf.named_steps['model'].coef_[0]
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})

print('Top features pushing toward Denied:')
display(coef_df.sort_values('coefficient', ascending=False).head(15))

print('Top features pushing toward Approved:')
display(coef_df.sort_values('coefficient').head(15))

# 14. Write-up Summary

## Dataset being used

The dataset used for Part 1 is `authsignal_denial_risk_dataset_200000.csv`, a synthetic healthcare prior-authorization dataset with 200,000 records and structured features describing request type, payer type, document availability, policy-match score, note completeness score, and denial outcome.

## Minimum two models being implemented in the project

1. **Denial Risk Predictor** using Logistic Regression, Random Forest, or Gradient Boosting.
2. **Denial Reason Classifier** using TF-IDF + Logistic Regression or TF-IDF + Linear SVM.

Optional third model: **Document Completeness Predictor** using Logistic Regression, Decision Tree, or Random Forest.

## Why these models are suitable

- Logistic Regression is suitable as a baseline because it is simple, fast, interpretable, and works well for binary classification.
- Random Forest or Gradient Boosting is suitable because denial risk may depend on non-linear combinations of missing documents, policy match, payer strictness, and request type.
- TF-IDF + Logistic Regression or Linear SVM is suitable for denial reason classification because denial letters are text documents and these algorithms are strong baselines for text classification tasks.